# VisoMaster Setup Notebook

This notebook handles the setup procedures for VisoMaster:
1. Installing required dependencies
2. Fixing Models.py backslash path issue
3. Downloading models
4. Installing TensorRT for acceleration
5. Replacing inswapper model with a compatible version

Run the cells in order to set up the environment. All operations are logged for debugging.

In [ ]:
# Setup logging system
import logging
import os
import sys
import datetime
import traceback

# Create logs directory if it doesn't exist
logs_dir = os.path.join(os.getcwd(), 'Logs')
os.makedirs(logs_dir, exist_ok=True)

# Configure logging
log_filename = os.path.join(logs_dir, f'setup_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger('visomaster_setup')

logger.info("=== VisoMaster Setup Started ===")
logger.info(f"Log file created at: {log_filename}")
logger.info(f"Current working directory: {os.getcwd()}")

In [ ]:
# Error handling decorator
def log_step(func):
    def wrapper(*args, **kwargs):
        step_name = func.__name__
        logger.info(f"Starting step: {step_name}")
        try:
            result = func(*args, **kwargs)
            logger.info(f"Successfully completed step: {step_name}")
            return result
        except Exception as e:
            logger.error(f"Error in step {step_name}: {str(e)}")
            logger.error(traceback.format_exc())
            raise
    return wrapper

In [ ]:
# Install scikit-image with conda
@log_step
def install_scikit_image():
    import subprocess
    logger.info("Installing scikit-image with conda...")
    result = subprocess.run(['conda', 'install', '-y', 'scikit-image'], 
                           capture_output=True, text=True)
    
    if result.returncode != 0:
        logger.error(f"Failed to install scikit-image: {result.stderr}")
        raise RuntimeError("Failed to install scikit-image")
        
    logger.info(result.stdout)
    return "✅ scikit-image installed"

print(install_scikit_image())

In [ ]:
# Install dependencies from requirements_cu124.txt
@log_step
def install_requirements():
    import subprocess
    # First check if we're in the VisoMaster directory
    current_dir = os.getcwd()
    logger.info(f"Current directory: {current_dir}")
    
    if 'visomaster' in current_dir.lower():
        visomaster_dir = current_dir
    else:
        visomaster_dir = '/workspace/visomaster'
        logger.info(f"Changing directory to: {visomaster_dir}")
        os.chdir(visomaster_dir)
    
    # Check if requirements file exists
    req_file = os.path.join(visomaster_dir, 'requirements_cu124.txt')
    if not os.path.exists(req_file):
        logger.error(f"Requirements file not found: {req_file}")
        raise FileNotFoundError(f"Requirements file not found: {req_file}")
    
    logger.info(f"Installing dependencies from: {req_file}")
    result = subprocess.run(['pip', 'install', '-r', req_file, '--no-cache-dir'],
                           capture_output=True, text=True)
    
    if result.returncode != 0:
        logger.error(f"Failed to install requirements: {result.stderr}")
        raise RuntimeError("Failed to install requirements")
        
    logger.info("Requirements installation completed")
    return f"✅ Dependencies from {req_file} installed"

print(install_requirements())

In [ ]:
# Download models
@log_step
def download_models():
    import subprocess
    # Navigate to model_assets directory
    current_dir = os.getcwd()
    if 'visomaster' in current_dir.lower():
        visomaster_dir = current_dir
    else:
        visomaster_dir = '/workspace/visomaster'
    
    model_assets_dir = os.path.join(visomaster_dir, 'model_assets')
    logger.info(f"Changing directory to: {model_assets_dir}")
    
    if not os.path.exists(model_assets_dir):
        logger.error(f"Model assets directory not found: {model_assets_dir}")
        raise FileNotFoundError(f"Model assets directory not found: {model_assets_dir}")
    
    os.chdir(model_assets_dir)
    
    # Check if download_models.py exists
    if not os.path.exists('download_models.py'):
        logger.error("download_models.py script not found")
        raise FileNotFoundError("download_models.py script not found")
    
    logger.info("Running download_models.py script...")
    result = subprocess.run(['python', 'download_models.py'],
                           capture_output=True, text=True)
    
    if result.returncode != 0:
        logger.error(f"Failed to download models: {result.stderr}")
        raise RuntimeError("Failed to download models")
    
    logger.info(result.stdout)
    logger.info("Models download completed")
    return "✅ Models downloaded"

print(download_models())

In [ ]:
# Download custom inswapper model for TensorRT compatibility
@log_step
def download_special_inswapper():
    import requests
    import shutil
    
    # Determine the model_assets directory
    current_dir = os.getcwd()
    if 'model_assets' in current_dir.lower():
        model_dir = current_dir
    elif 'visomaster' in current_dir.lower():
        model_dir = os.path.join(current_dir, 'model_assets')
    else:
        model_dir = '/workspace/visomaster/model_assets'
    
    # Move to model directory
    if not os.path.exists(model_dir):
        logger.error(f"Model directory not found: {model_dir}")
        raise FileNotFoundError(f"Model directory not found: {model_dir}")
    
    os.chdir(model_dir)
    logger.info(f"Changed directory to: {model_dir}")
    
    # URL for the custom inswapper model
    inswapper_url = "https://huggingface.co/Red1618/Viso/resolve/main/inswapper_128_fp16.onnx?download=true"
    output_file = os.path.join(model_dir, 'inswapper_128_fp16.onnx')
    output_renamed = os.path.join(model_dir, 'inswapper_128.fp16.onnx')
    
    # Download the file
    logger.info(f"Downloading special inswapper model from: {inswapper_url}")
    try:
        response = requests.get(inswapper_url, stream=True)
        response.raise_for_status()  # Check if download was successful
        
        with open(output_file, 'wb') as f:
            shutil.copyfileobj(response.raw, f)
        
        # Rename the file
        if os.path.exists(output_renamed):
            logger.info(f"Removing existing file: {output_renamed}")
            os.remove(output_renamed)
            
        os.rename(output_file, output_renamed)
        logger.info(f"Renamed {output_file} to {output_renamed}")
        
        # Check file size to make sure it's valid
        file_size = os.path.getsize(output_renamed) / (1024 * 1024)  # Size in MB
        logger.info(f"Downloaded file size: {file_size:.2f} MB")
        
        if file_size < 1:  # If file is less than 1MB, it's probably not correct
            logger.warning("Downloaded file is suspiciously small. May not be valid.")
        
        return f"✅ Special inswapper model downloaded and renamed to {os.path.basename(output_renamed)}"
    except Exception as e:
        logger.error(f"Failed to download inswapper model: {str(e)}")
        raise

print(download_special_inswapper())

In [ ]:
# Create dependencies folder and download ffmpeg
@log_step
def setup_dependencies():
    import requests
    import shutil
    import tarfile
    import zipfile
    
    # Determine the visomaster directory
    current_dir = os.getcwd()
    if 'visomaster' in current_dir.lower():
        visomaster_dir = current_dir
    else:
        visomaster_dir = '/workspace/visomaster'
    
    # Create dependencies directory
    deps_dir = os.path.join(visomaster_dir, 'dependencies')
    os.makedirs(deps_dir, exist_ok=True)
    logger.info(f"Created dependencies directory: {deps_dir}")
    
    # Download FFMPEG
    ffmpeg_url = "https://github.com/visomaster/visomaster-assets/releases/download/v0.1.0_dp/ffmpeg.exe"
    ffmpeg_file = os.path.join(deps_dir, 'ffmpeg.exe')
    
    logger.info(f"Downloading FFMPEG from: {ffmpeg_url}")
    try:
        response = requests.get(ffmpeg_url, stream=True)
        response.raise_for_status()
        
        with open(ffmpeg_file, 'wb') as f:
            shutil.copyfileobj(response.raw, f)
        
        # Make the file executable
        os.chmod(ffmpeg_file, 0o755)  # rwxr-xr-x
        logger.info(f"FFMPEG downloaded to: {ffmpeg_file}")
        
        return "✅ FFMPEG downloaded and set up in dependencies folder"
    except Exception as e:
        logger.error(f"Failed to download FFMPEG: {str(e)}")
        raise

print(setup_dependencies())

In [ ]:
# Download and install TensorRT
@log_step
def setup_tensorrt():
    import requests
    import shutil
    import tarfile
    import subprocess
    
    # Move to workspace directory
    workspace_dir = '/workspace'
    os.chdir(workspace_dir)
    logger.info(f"Changed directory to: {workspace_dir}")
    
    # Download TensorRT
    tensorrt_url = "https://huggingface.co/Red1618/Viso/resolve/main/TensorRT-10.1.0.27.Linux.x86_64-gnu.cuda-12.4.tar.gz?download=true"
    tensorrt_file = os.path.join(workspace_dir, 'tensorrt.tar.gz')
    
    logger.info(f"Downloading TensorRT from: {tensorrt_url}")
    try:
        response = requests.get(tensorrt_url, stream=True)
        response.raise_for_status()
        
        with open(tensorrt_file, 'wb') as f:
            shutil.copyfileobj(response.raw, f)
        
        # Extract TensorRT
        tensorrt_dir = os.path.join(workspace_dir, 'tensorrt')
        os.makedirs(tensorrt_dir, exist_ok=True)
        
        logger.info(f"Extracting TensorRT to: {tensorrt_dir}")
        with tarfile.open(tensorrt_file) as tar:
            tar.extractall(path=tensorrt_dir)
        
        # Clean up
        os.remove(tensorrt_file)
        logger.info("Removed TensorRT archive file")
        
        # Set up environment variables
        os.environ['LD_LIBRARY_PATH'] = f"{tensorrt_dir}/lib:{os.environ.get('LD_LIBRARY_PATH', '')}"
        os.environ['PATH'] = f"{tensorrt_dir}/bin:{os.environ.get('PATH', '')}"
        
        logger.info(f"Updated LD_LIBRARY_PATH: {os.environ['LD_LIBRARY_PATH']}")
        logger.info(f"Updated PATH: {os.environ['PATH']}")
        
        # Install TensorRT Python package
        python_dir = os.path.join(tensorrt_dir, 'python')
        if os.path.exists(python_dir):
            os.chdir(python_dir)
            logger.info(f"Installing TensorRT Python package from: {python_dir}")
            
            # Find the appropriate wheel file for Python 3.10
            wheel_file = None
            for file in os.listdir(python_dir):
                if file.startswith('tensorrt') and 'cp310' in file and file.endswith('.whl'):
                    wheel_file = file
                    break
            
            if wheel_file:
                result = subprocess.run(['pip', 'install', wheel_file],
                                       capture_output=True, text=True)
                if result.returncode != 0:
                    logger.error(f"Failed to install TensorRT Python package: {result.stderr}")
                else:
                    logger.info("TensorRT Python package installed")
            else:
                logger.warning("No compatible TensorRT wheel found for Python 3.10")
        else:
            logger.warning(f"TensorRT Python directory not found: {python_dir}")
        
        return "✅ TensorRT downloaded and installed"
    except Exception as e:
        logger.error(f"Failed to set up TensorRT: {str(e)}")
        raise

print(setup_tensorrt())

In [ ]:
# Verify installation
@log_step
def verify_installation():
    # Determine paths
    current_dir = os.getcwd()
    if 'model_assets' in current_dir.lower():
        model_dir = current_dir
        visomaster_dir = os.path.dirname(current_dir)
    elif 'visomaster' in current_dir.lower():
        visomaster_dir = current_dir
        model_dir = os.path.join(visomaster_dir, 'model_assets')
    else:
        visomaster_dir = '/workspace/visomaster'
        model_dir = os.path.join(visomaster_dir, 'model_assets')
    
    # Check if model directory exists
    if not os.path.exists(model_dir):
        logger.error(f"Model directory not found: {model_dir}")
        raise FileNotFoundError(f"Model directory not found: {model_dir}")
    
    # Check if model files exist
    model_files = os.listdir(model_dir)
    logger.info(f"Found {len(model_files)} files in model directory")
    
    for file in model_files:
        file_path = os.path.join(model_dir, file)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # Size in MB
        logger.info(f"Model file: {file}, Size: {file_size:.2f} MB")
    
    # Check if TensorRT is installed
    logger.info("Checking TensorRT installation...")
    try:
        import tensorrt as trt
        logger.info(f"TensorRT version: {trt.__version__}")
    except ImportError:
        logger.warning("TensorRT is not properly installed or not in Python path")
    
    # Check for custom inswapper model
    inswapper_path = os.path.join(model_dir, 'inswapper_128.fp16.onnx')
    if os.path.exists(inswapper_path):
        file_size = os.path.getsize(inswapper_path) / (1024 * 1024)  # Size in MB
        logger.info(f"Custom inswapper model found, size: {file_size:.2f} MB")
    else:
        logger.warning("Custom inswapper model not found!")
    
    # Check FFMPEG
    deps_dir = os.path.join(visomaster_dir, 'dependencies')
    ffmpeg_path = os.path.join(deps_dir, 'ffmpeg.exe')
    if os.path.exists(ffmpeg_path):
        logger.info(f"FFMPEG found at: {ffmpeg_path}")
    else:
        logger.warning(f"FFMPEG not found at: {ffmpeg_path}")
    
    logger.info("Installation verification completed")
    return "\n✅ Setup complete! VisoMaster is ready to use with TensorRT acceleration."

print(verify_installation())